In [ ]:

library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(car)
library(MuMIn)
library(marginaleffects)
library(tibble)
library(openxlsx)   

df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness_segmented_2004.csv")
df <- df %>%
  mutate(
    SiteID   = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone),
    period   = as.character(period),
    Period_bin = ifelse(period == "2005_2018", 1, 0)
  )

num_cols <- c(
  "sen_richness","sen_temp","HFP_period",
  "mean_temp","mean_Q","mean_salinity",
  "mean_organic","elevation"
)
df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      sen_richness, sen_temp, HFP_period,
      mean_temp, mean_Q, mean_salinity,
      mean_organic, elevation,
      Protocol, zone, HYBAS_ID, SiteID
    )
  )

vars <- c(
  "mean_temp","sen_temp","mean_Q",
  "mean_organic","mean_salinity",
  "HFP_period","elevation"
)

for (v in vars) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[, 1]
}

fixed_form <- sen_richness ~
  Period_bin +
  mean_temp_z + sen_temp_z + mean_Q_z +
  mean_organic_z + mean_salinity_z +
  HFP_period_z + elevation_z +
  mean_temp_z:Period_bin +
  sen_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_organic_z:Period_bin +
  mean_salinity_z:Period_bin +
  HFP_period_z:Period_bin +
  elevation_z:Period_bin

form <- update(
  fixed_form,
  . ~ . + (1 | HYBAS_ID) + (1 | Protocol) + (1 | SiteID)
)

zones <- c("All", "Cold", "Intermediate", "Warm")
vif_threshold <- 10
core_vars <- c("mean_temp", "sen_temp")  

get_vif_vals <- function(dat) {
  lm_fit <- lm(fixed_form, data = dat)
  vif_raw <- car::vif(lm_fit)
  if (is.matrix(vif_raw)) {
    vif_raw[, "GVIF^(1/(2*Df))"]
  } else {
    vif_raw
  }
}

is_high_vif <- function(v, vif_vals) {
  if (v %in% core_vars) return(FALSE)

  terms <- c(
    paste0(v, "_z"),
    paste0(v, "_z:Period_bin"),
    paste0("Period_bin:", v, "_z")
  )
  terms <- terms[terms %in% names(vif_vals)]
  if (length(terms) == 0) return(FALSE)

  any(vif_vals[terms] >= vif_threshold)
}

sig_label <- function(p) {
  if (is.na(p)) "" else if (p < 0.001) "***"
  else if (p < 0.01) "**"
  else if (p < 0.05) "*"
  else ""
}
effects_all <- list()
model_stats <- list()
vif_tables  <- list()
for (z in zones) {

  dat <- if (z == "All") df else filter(df, zone == z)
  if (nrow(dat) < 50) next

  cat("Running zone:", z, "\n")

  # ---------------- VIF ----------------
  vif_vals <- get_vif_vals(dat)
  vif_tables[[z]] <- data.frame(
    Zone = z,
    Term = names(vif_vals),
    VIF  = as.numeric(vif_vals),
    row.names = NULL
  )
  high_vif <- sapply(vars, is_high_vif, vif_vals = vif_vals)
  names(high_vif) <- vars

  # ---------------- Model ----------------
  m <- lmer(
    form, data = dat, REML = TRUE,
    control = lmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
  )

  # ---------------- Save full summary ----------------
  writeLines(
    capture.output(summary(m)),
    paste0("C:/Users/Lenovo/Phase-specific modelling of richness trends/summary_Richness_", z, ".txt")
  )

  # ---------------- Save fixed effects ----------------
  coef_tab <- coef(summary(m)) %>%
    as.data.frame() %>%
    rownames_to_column("Term") %>%
    mutate(Zone = z)

  write_csv(
    coef_tab,
    paste0("C:/Users/Lenovo/Phase-specific modelling of richness trends/coef_Richness_", z, ".csv")
  )

  # ---------------- Save random effects ----------------
  vc_tab <- as.data.frame(VarCorr(m)) %>%
    mutate(Zone = z)

  write_csv(
    vc_tab,
    paste0("C:/Users/Lenovo/Phase-specific modelling of richness trends/varcomp_Richness_", z, ".csv")
  )

  # ---------------- Fit statistics ----------------
  r2 <- suppressWarnings(MuMIn::r.squaredGLMM(m))
  model_stats[[z]] <- data.frame(
    Zone = z,
    AIC = AIC(m),
    BIC = BIC(m),
    R2_marginal = r2[1],
    R2_conditional = r2[2],
    row.names = NULL
  )

  cf <- coef(summary(m))

  # ---------------- Marginal effects ----------------
  for (v in vars) {

    if (isTRUE(high_vif[[v]])) {
      effects_all[[length(effects_all) + 1]] <- data.frame(
        Zone = z, Factor = v,

        Early_Effect = NA, Early_CI95_L = NA, Early_CI95_U = NA,
        Early_p = NA, Early_sig = "",

        Late_Effect = NA, Late_CI95_L = NA, Late_CI95_U = NA,
        Late_p = NA, Late_sig = "",

        Delta = NA,
        Delta_CI95_L = NA, Delta_CI95_U = NA,
        Delta_p = NA, Delta_sig = "",
        row.names = NULL
      )
      next
    }

    vz <- paste0(v, "_z")

    # ---- Early AME ----
    ame_e <- avg_slopes(
      m, variables = vz,
      newdata = dat %>% mutate(Period_bin = 0)
    )
    b_e  <- ame_e$estimate[1]
    ci_e <- c(ame_e$conf.low[1], ame_e$conf.high[1])
    p_e  <- ame_e$p.value[1]

    # ---- Late AME ----
    ame_l <- avg_slopes(
      m, variables = vz,
      newdata = dat %>% mutate(Period_bin = 1)
    )
    b_l  <- ame_l$estimate[1]
    ci_l <- c(ame_l$conf.low[1], ame_l$conf.high[1])
    p_l  <- ame_l$p.value[1]

    # ---- Delta (Late - Early) ----
    b_i <- b_l - b_e

    # Delta significance = interaction term significance
    inter1 <- paste0(vz, ":Period_bin")
    inter2 <- paste0("Period_bin:", vz)
    inter  <- if (inter1 %in% rownames(cf)) inter1 else inter2

    delta_est <- if (inter %in% rownames(cf)) cf[inter, "Estimate"] else NA
    delta_se  <- if (inter %in% rownames(cf)) cf[inter, "Std. Error"] else NA
    p_d       <- if (inter %in% rownames(cf)) cf[inter, "Pr(>|t|)"] else NA

    df_d <- if (inter %in% rownames(cf)) cf[inter, "df"] else NA
    tcrit <- if (!is.na(df_d)) qt(0.975, df = df_d) else 1.96

    delta_ciL <- if (!is.na(delta_est) && !is.na(delta_se))
      delta_est - tcrit * delta_se else NA
    delta_ciU <- if (!is.na(delta_est) && !is.na(delta_se))
      delta_est + tcrit * delta_se else NA


    effects_all[[length(effects_all) + 1]] <- data.frame(
      Zone = z, Factor = v,

      Early_Effect = b_e,
      Early_CI95_L = ci_e[1], Early_CI95_U = ci_e[2],
      Early_p = p_e, Early_sig = sig_label(p_e),

      Late_Effect = b_l,
      Late_CI95_L = ci_l[1], Late_CI95_U = ci_l[2],
      Late_p = p_l, Late_sig = sig_label(p_l),

      Delta = b_i,
      Delta_CI95_L = delta_ciL, Delta_CI95_U = delta_ciU,
      Delta_p = p_d, Delta_sig = sig_label(p_d),
      row.names = NULL
    )
  }
}

# =========================================================
# Output (Dataset S2)
# =========================================================
write_csv(bind_rows(effects_all),
          "C:/Users/Lenovo/Phase-specific modelling of richness trends/Period_MarginalEffects_by_zone.csv")

write_csv(bind_rows(model_stats),
          "C:/Users/Lenovo/Phase-specific modelling of richness trends/Model_fit_stats.csv")

Warning message:
"package 'lme4' was built under R version 4.5.2"
Loading required package: Matrix

Warning message:
"package 'lmerTest' was built under R version 4.5.2"

Attaching package: 'lmerTest'


The following object is masked from 'package:lme4':

    lmer


The following object is masked from 'package:stats':

    step


Warning message:
"package 'dplyr' was built under R version 4.5.2"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'readr' was built under R version 4.5.2"
Warning message:
"package 'car' was built under R version 4.5.2"
Loading required package: carData

Warning message:
"package 'carData' was built under R version 4.5.2"

Attaching package: 'car'


The following object is masked from 'package:dplyr':

    recode


Warning message:
"package 'MuMIn' was built under R version 4.5.2"
Wa

Running zone: All 


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif

boundary (singular) fit: see help('isSingular')


Correlation matrix not shown by default, as p = 16 > 12.
Use print(out$value, correlation=TRUE)  or
    vcov(out$value)        if you need it


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `mar

Running zone: Cold 


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif

boundary (singular) fit: see help('isSingular')


Correlation matrix not shown by default, as p = 16 > 12.
Use print(out$value, correlation=TRUE)  or
    vcov(out$value)        if you need it


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `mar

Running zone: Intermediate 


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif

boundary (singular) fit: see help('isSingular')


Correlation matrix not shown by default, as p = 16 > 12.
Use print(out$value, correlation=TRUE)  or
    vcov(out$value)        if you need it


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `mar

Running zone: Warm 


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif


Correlation matrix not shown by default, as p = 16 > 12.
Use print(out$value, correlation=TRUE)  or
    vcov(out$value)        if you need it


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncerta

In [ ]:
# =========================================================
# Spatial autocorrelation diagnostics of model residuals (Dataset S2)
# =========================================================
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(spdep)


df <- read_csv(
  "D:/NC/Data/rivernet/inputdata/Richness_segmented_2004.csv"
)

df <- df %>%
  mutate(
    SiteID = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Period_bin = ifelse(period == "2005_2018", 1, 0)
  )

scale_cols <- c(
  "sen_temp","HFP_period","mean_temp",
  "mean_Q","mean_salinity","mean_organic","elevation"
)
for (v in scale_cols) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[,1]
}

form <- sen_richness ~
  Period_bin +
  sen_temp_z + HFP_period_z + mean_temp_z + mean_Q_z +
  mean_salinity_z + mean_organic_z + elevation_z +
  sen_temp_z:Period_bin +
  HFP_period_z:Period_bin +
  mean_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_salinity_z:Period_bin +
  mean_organic_z:Period_bin +
  elevation_z:Period_bin +
  (1 | HYBAS_ID) + (1 | Protocol) + (1 | SiteID)

zones <- c("All","Cold","Intermediate","Warm")

out <- list()

for (z in zones) {

  dat <- if (z == "All") df else filter(df, zone == z)
  m <- lmer(form, data = dat, REML = TRUE)
  mf <- model.frame(m)
  mf$resid <- resid(m)
  dat_used <- dat[as.numeric(rownames(mf)), ]

  mf$SiteID    <- dat_used$SiteID
  mf$Longitude <- dat_used$Longitude
  mf$Latitude  <- dat_used$Latitude
  site_res <- mf %>%
    group_by(SiteID) %>%
    summarise(
      resid = mean(resid, na.rm = TRUE),
      Longitude = first(Longitude),
      Latitude  = first(Latitude),
      .groups = "drop"
    )

  coords <- as.matrix(site_res[, c("Longitude", "Latitude")])

  knn <- knearneigh(coords, k = 10)
  nb  <- knn2nb(knn)
  lw  <- nb2listw(nb, style = "W", zero.policy = TRUE)

  mi <- moran.test(
    site_res$resid,
    lw,
    randomisation = TRUE,
    zero.policy = TRUE
  )

  out[[z]] <- data.frame(
    Zone = z,
    N_sites = nrow(site_res),
    Moran_I = unname(mi$estimate["Moran I statistic"]),
    P_value = mi$p.value
  )
}

df_moran <- bind_rows(out)

write_csv(
  df_moran,
  "C:/Users/Lenovo/Phase-specific modelling of richness trends/MoranI_residuals_by_zone_SITE_Richness.csv"
)
print(df_moran)

Warning message:
"package 'spdep' was built under R version 4.5.2"
Loading required package: spData

Warning message:
"package 'spData' was built under R version 4.5.2"
To access larger datasets in this package, install the spDataLarge
package with: `install.packages('spDataLarge',
repos='https://nowosad.github.io/drat/', type='source')`

Loading required package: sf

Linking to GEOS 3.13.1, GDAL 3.11.0, PROJ 9.6.0; sf_use_s2() is TRUE

Rows: 6834 Columns: 18
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (4): SiteID, period, Protocol, zone
dbl (14): sen_richness, p_richness, mean_salinity, mean_organic, seg_id_nat,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
boundary (singular) fit: see help('isSingular')

Warning message in knn2nb(knn):
"neighbour object has 5 sub-graphs"
boundary (singular) fit: see help('isSingular')

          Zone N_sites     Moran_I      P_value
1          All    3346  0.03453776 8.077363e-07
2         Cold     664 -0.01414696 7.873226e-01
3 Intermediate    1954  0.05986158 7.760109e-11
4         Warm     728 -0.01188657 7.509339e-01


In [ ]:
# =========================================================
#  random-structure comparison (ML) (Dataset S7)
# =========================================================
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(MuMIn)
library(marginaleffects)
library(tibble)

# =========================================================
# Read data
# =========================================================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness_segmented_2004.csv") %>%
  mutate(
    SiteID   = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone),
    period   = as.character(period),
    Period_bin = ifelse(period == "2005_2018", 1, 0)
  )

num_cols <- c(
  "sen_richness","sen_temp","HFP_period",
  "mean_temp","mean_Q","mean_salinity",
  "mean_organic","elevation"
)
df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>% filter(
  complete.cases(
    sen_richness, sen_temp, HFP_period,
    mean_temp, mean_Q, mean_salinity,
    mean_organic, elevation,
    Protocol, zone, HYBAS_ID, SiteID
  )
)

# =========================================================
# Global Z-score
# =========================================================
vars <- c(
  "mean_temp","sen_temp","mean_Q",
  "mean_organic","mean_salinity",
  "HFP_period","elevation"
)
for (v in vars) df[[paste0(v, "_z")]] <- scale(df[[v]])[,1]

# =========================================================
# Fixed effects formula
# =========================================================
fixed_form <- sen_richness ~
  Period_bin +
  mean_temp_z + sen_temp_z + mean_Q_z +
  mean_organic_z + mean_salinity_z +
  HFP_period_z + elevation_z +
  mean_temp_z:Period_bin +
  sen_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_organic_z:Period_bin +
  mean_salinity_z:Period_bin +
  HFP_period_z:Period_bin +
  elevation_z:Period_bin

zones <- c("All","Cold","Intermediate","Warm")

# =========================================================
# Candidate random-structure specifications
# =========================================================
specs <- list(
  M1 = ". ~ . + (1|HYBAS_ID) + (1|Protocol)",              # baseline
  M2 = ". ~ . + (1|HYBAS_ID) + (1|Protocol) + (1|SiteID)", # +SiteID
  M3 = ". ~ . + (1|HYBAS_ID) + C(Protocol)",               # Protocol fixed
  M4 = ". ~ . + (1|HYBAS_ID) + C(Protocol) + (1|SiteID)",  # fixed + SiteID
  M5 = ". ~ . + (1|Protocol) + (1|SiteID)",                # drop HYBAS
  M6 = ". ~ . + C(Protocol) + (1|SiteID)"                  # drop HYBAS, fixed Protocol
)

# =========================================================
# Helper: extract ALL fixed effects + Delta significance
# =========================================================
get_all_terms <- function(m){
  cf <- coef(summary(m)) %>%
    as.data.frame() %>%
    rownames_to_column("Term")

  cf <- cf %>%
    filter(!Term %in% "(Intercept)") %>%
    mutate(
      Effect_type = case_when(
        grepl(":Period_bin", Term) | grepl("Period_bin:", Term) ~ "Delta",
        Term == "Period_bin" ~ "Period",
        TRUE ~ "Main"
      ),
      Delta_var = ifelse(
        Effect_type == "Delta",
        gsub(":Period_bin|Period_bin:", "", Term),
        NA
      )
    )

  cf
}

# =========================================================
# Helper: AME + Delta 
# =========================================================
get_ame_all <- function(m, dat){
  out <- list()

  for (v in vars){
    vz <- paste0(v, "_z")

    a0 <- avg_slopes(m, variables = vz,
                     newdata = dat %>% mutate(Period_bin=0))
    a1 <- avg_slopes(m, variables = vz,
                     newdata = dat %>% mutate(Period_bin=1))

    out[[v]] <- data.frame(
      Variable = v,
      Early_Effect = a0$estimate[1],
      Early_p = a0$p.value[1],
      Late_Effect  = a1$estimate[1],
      Late_p  = a1$p.value[1],
      Delta = a1$estimate[1] - a0$estimate[1]
    )
  }
  bind_rows(out)
}

# =========================================================
# Containers
# =========================================================
fit_rows  <- list()
coef_rows <- list()
ame_rows  <- list()

# =========================================================
# Run sensitivity analysis
# =========================================================
for (z in zones){

  dat <- if (z=="All") df else filter(df, zone==z)
  if (nrow(dat) < 50) next

  cat("\n========================\nZone:", z, " N=", nrow(dat), "\n")

  for (nm in names(specs)){

    form <- update(fixed_form, as.formula(specs[[nm]]))

    m <- tryCatch(
      lmer(
        form, data=dat, REML=FALSE,
        control=lmerControl(
          optimizer="bobyqa",
          optCtrl=list(maxfun=2e5)
        )
      ),
      error=function(e) NULL
    )
    if (is.null(m)) next

    # ---- model fit ----
    r2 <- suppressWarnings(MuMIn::r.squaredGLMM(m))
    vc <- as.data.frame(VarCorr(m))

    fit_rows[[length(fit_rows)+1]] <- data.frame(
      Zone=z, Model=nm,
      N=nrow(dat),
      AIC=AIC(m), BIC=BIC(m),
      R2_marginal=r2[1], R2_conditional=r2[2],
      Singular=isSingular(m),
      Var_HYBAS = vc$vcov[vc$grp=="HYBAS_ID"][1] %||% NA,
      Var_Protocol = vc$vcov[vc$grp=="Protocol"][1] %||% NA,
      Var_SiteID = vc$vcov[vc$grp=="SiteID"][1] %||% NA,
      Var_Residual = vc$vcov[vc$grp=="Residual"][1] %||% NA
    )

    # ---- ALL fixed effects + Delta p ----
    cf_all <- get_all_terms(m) %>%
      mutate(Zone=z, Model=nm)

    coef_rows[[length(coef_rows)+1]] <- cf_all

    # ---- AME (all vars) ----
    ame <- get_ame_all(m, dat) %>%
      mutate(Zone=z, Model=nm)

    ame_rows[[length(ame_rows)+1]] <- ame
  }
}

`%||%` <- function(x, y) if (length(x)==0) y else x

# =========================================================
# Output
# =========================================================
write_csv(bind_rows(fit_rows),
          "C:/Users/Lenovo/Phase-specific modelling of richness trends/SensA_ModelCompare_ML.csv")

write_csv(bind_rows(coef_rows),
          "C:/Users/Lenovo/Phase-specific modelling of richness trends/SensA_AllFixedEffects_ML.csv")

write_csv(bind_rows(ame_rows),
          "C:/Users/Lenovo/Phase-specific modelling of richness trends/SensA_AME_allvars_ML.csv")

Rows: 6834 Columns: 18
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (4): SiteID, period, Protocol, zone
dbl (14): sen_richness, p_richness, mean_salinity, mean_organic, seg_id_nat,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.



Zone: All  N= 5215 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa


Zone: Cold  N= 1073 


boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe 


Zone: Intermediate  N= 3079 


boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe 


Zone: Warm  N= 1063 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa

In [ ]:
# =========================================================
# Sensitivity S1: Outlier influence (trim top X% abs residuals) (Dataset S9)
# =========================================================
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(MuMIn)
library(marginaleffects)
library(tibble)

# =========================
# User settings
# =========================
trim_prop <- 0.01      # top 1% |resid|
zones <- c("All","Cold","Intermediate","Warm")

# =========================
# Read data
# =========================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness_segmented_2004.csv") %>%
  mutate(
    SiteID   = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone),
    period   = as.character(period),
    Period_bin = ifelse(period == "2005_2018", 1, 0)
  )

num_cols <- c(
  "sen_richness","sen_temp","HFP_period",
  "mean_temp","mean_Q","mean_salinity",
  "mean_organic","elevation"
)
df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      sen_richness, sen_temp, HFP_period,
      mean_temp, mean_Q, mean_salinity,
      mean_organic, elevation,
      Protocol, zone, HYBAS_ID, SiteID
    )
  )

# =========================
# Global Z-score
# =========================
vars <- c(
  "mean_temp","sen_temp","mean_Q",
  "mean_organic","mean_salinity",
  "HFP_period","elevation"
)
for (v in vars) df[[paste0(v, "_z")]] <- scale(df[[v]])[,1]

# =========================
# Model
# =========================
fixed_form <- sen_richness ~
  Period_bin +
  mean_temp_z + sen_temp_z + mean_Q_z +
  mean_organic_z + mean_salinity_z +
  HFP_period_z + elevation_z +
  mean_temp_z:Period_bin +
  sen_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_organic_z:Period_bin +
  mean_salinity_z:Period_bin +
  HFP_period_z:Period_bin +
  elevation_z:Period_bin

form_main <- update(
  fixed_form,
  . ~ . + (1|HYBAS_ID) + (1|Protocol) + (1|SiteID)
)

# =========================
# ---- extract Delta p + CI from interaction term ----
get_delta_stats <- function(m, v){
  vz <- paste0(v, "_z")
  cf <- coef(summary(m))

  inter1 <- paste0(vz, ":Period_bin")
  inter2 <- paste0("Period_bin:", vz)
  inter  <- if (inter1 %in% rownames(cf)) inter1 else inter2

  if (!inter %in% rownames(cf)) {
    return(c(NA, NA, NA))
  }

  est <- cf[inter, "Estimate"]
  se  <- cf[inter, "Std. Error"]
  df  <- cf[inter, "df"]
  p   <- cf[inter, "Pr(>|t|)"]

  tcrit <- qt(0.975, df = df)

  c(
    Delta_p = p,
    Delta_CI_L = est - tcrit * se,
    Delta_CI_U = est + tcrit * se
  )
}

# ---- AME + Delta ----
get_ame_allvars <- function(m, dat, z, tag){
  out <- list()

  for (v in vars){
    vz <- paste0(v, "_z")

    a0 <- avg_slopes(m, variables=vz, newdata = dat %>% mutate(Period_bin=0))
    a1 <- avg_slopes(m, variables=vz, newdata = dat %>% mutate(Period_bin=1))

    delta_stats <- get_delta_stats(m, v)

    out[[v]] <- data.frame(
      Zone=z, Tag=tag, Factor=v,

      Early_Effect=a0$estimate[1],
      Early_CI_L=a0$conf.low[1], Early_CI_U=a0$conf.high[1],
      Early_p=a0$p.value[1],

      Late_Effect=a1$estimate[1],
      Late_CI_L=a1$conf.low[1], Late_CI_U=a1$conf.high[1],
      Late_p=a1$p.value[1],

      Delta=a1$estimate[1] - a0$estimate[1],
      Delta_p=delta_stats["Delta_p"],
      Delta_CI_L=delta_stats["Delta_CI_L"],
      Delta_CI_U=delta_stats["Delta_CI_U"]
    )
  }
  bind_rows(out)
}

# =========================
# Run S1
# =========================
sum_rows <- list()
ame_rows <- list()

for (z in zones){

  dat0 <- if (z=="All") df else filter(df, zone==z)
  if (nrow(dat0) < 50) next

  cat("\nRunning S1 | Zone:", z, " N=", nrow(dat0), "\n")

  m_full <- lmer(form_main, data=dat0, REML=TRUE,
                 control=lmerControl(optimizer="bobyqa", optCtrl=list(maxfun=2e5)))

  r <- residuals(m_full, type="pearson")
  cut <- quantile(abs(r), 1 - trim_prop, na.rm=TRUE)
  dat1 <- dat0[abs(r) <= cut, , drop=FALSE]

  m_trim <- lmer(form_main, data=dat1, REML=TRUE,
                 control=lmerControl(optimizer="bobyqa", optCtrl=list(maxfun=2e5)))

  r2_full <- suppressWarnings(MuMIn::r.squaredGLMM(m_full))
  r2_trim <- suppressWarnings(MuMIn::r.squaredGLMM(m_trim))

  sum_rows[[length(sum_rows)+1]] <- data.frame(
    Zone=z,
    N_full=nrow(dat0),
    N_trim=nrow(dat1),
    Trim_prop=1 - nrow(dat1)/nrow(dat0),
    R2m_full=r2_full[1], R2c_full=r2_full[2],
    R2m_trim=r2_trim[1], R2c_trim=r2_trim[2],
    AIC_full=AIC(m_full), AIC_trim=AIC(m_trim)
  )

  ame_rows[[length(ame_rows)+1]] <- get_ame_allvars(m_full, dat0, z, "Full")
  ame_rows[[length(ame_rows)+1]] <- get_ame_allvars(m_trim, dat1, z, "Trim")
}

# =========================
# Output
# =========================
write_csv(bind_rows(sum_rows),
          "C:/Users/Lenovo/Phase-specific modelling of richness trends/SensB_OutlierTrim_summary.csv")

write_csv(bind_rows(ame_rows),
          "C:/Users/Lenovo/Phase-specific modelling of richness trends/SensB_OutlierTrim_AME_allvars.csv")

Rows: 6834 Columns: 18
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (4): SiteID, period, Protocol, zone
dbl (14): sen_richness, p_richness, mean_salinity, mean_organic, seg_id_nat,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.



Running S1 | Zone: All  N= 5215 


boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some


Running S1 | Zone: Cold  N= 1073 


boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some


Running S1 | Zone: Intermediate  N= 3079 


boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some


Running S1 | Zone: Warm  N= 1063 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa

In [ ]:
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(car)
library(MuMIn)
library(marginaleffects)
library(tibble)

# =========================================================
# Read data
# =========================================================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness_segmented_2003.csv") %>%
  mutate(
    SiteID   = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone),
    period   = as.character(period),
    Period_bin = ifelse(period == "2004_2018", 1, 0)
  )

num_cols <- c(
  "sen_richness","sen_temp","HFP_period",
  "mean_temp","mean_Q","mean_salinity",
  "mean_organic","elevation"
)
df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      sen_richness, sen_temp, HFP_period,
      mean_temp, mean_Q, mean_salinity,
      mean_organic, elevation,
      Protocol, HYBAS_ID, SiteID
    )
  )

# =========================================================
# Standardize predictors
# =========================================================
vars <- c(
  "mean_temp","sen_temp","mean_Q",
  "mean_organic","mean_salinity",
  "HFP_period","elevation"
)

for (v in vars) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[,1]
}

# =========================================================
# Model formula
# =========================================================
fixed_form <- sen_richness ~
  Period_bin +
  mean_temp_z + sen_temp_z + mean_Q_z +
  mean_organic_z + mean_salinity_z +
  HFP_period_z + elevation_z +
  mean_temp_z:Period_bin +
  sen_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_organic_z:Period_bin +
  mean_salinity_z:Period_bin +
  HFP_period_z:Period_bin +
  elevation_z:Period_bin

form <- update(
  fixed_form,
  . ~ . + (1 | HYBAS_ID) + (1 | Protocol) + (1 | SiteID)
)

# =========================================================
# VIF
# =========================================================
core_vars <- c("mean_temp", "sen_temp")
vif_threshold <- 10

get_vif_vals <- function(dat) {
  lm_fit <- lm(fixed_form, data = dat)
  vif_raw <- car::vif(lm_fit)
  if (is.matrix(vif_raw)) {
    vif_raw[, "GVIF^(1/(2*Df))"]
  } else vif_raw
}

is_high_vif <- function(v, vif_vals) {
  if (v %in% core_vars) return(FALSE)
  terms <- c(
    paste0(v, "_z"),
    paste0(v, "_z:Period_bin"),
    paste0("Period_bin:", v, "_z")
  )
  terms <- terms[terms %in% names(vif_vals)]
  any(vif_vals[terms] >= vif_threshold)
}

sig_label <- function(p) {
  if (is.na(p)) "" else if (p < 0.001) "***"
  else if (p < 0.01) "**"
  else if (p < 0.05) "*"
  else ""
}

# =========================================================
# Run model
# =========================================================
cat("Running Richness model: ALL\n")

# ---- VIF ----
vif_vals <- get_vif_vals(df)
vif_table <- data.frame(
  Zone = "All",
  Term = names(vif_vals),
  VIF  = as.numeric(vif_vals)
)

high_vif <- sapply(vars, is_high_vif, vif_vals = vif_vals)
names(high_vif) <- vars

# ---- Mixed model ----
m <- lmer(
  form, data = df, REML = TRUE,
  control = lmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 2e5)
  )
)

# ---- Save summaries ----
writeLines(
  capture.output(summary(m)),
  "C:/Users/Lenovo/Phase-specific modelling of 2003/summary_Richness_All.txt"
)

coef_tab <- coef(summary(m)) %>%
  as.data.frame() %>%
  rownames_to_column("Term")

write_csv(
  coef_tab,
  "C:/Users/Lenovo/Phase-specific modelling of 2003/coef_Richness_All.csv"
)

vc_tab <- as.data.frame(VarCorr(m))
write_csv(
  vc_tab,
  "C:/Users/Lenovo/Phase-specific modelling of 2003/varcomp_Richness_All.csv"
)

r2 <- suppressWarnings(MuMIn::r.squaredGLMM(m))
model_stats <- data.frame(
  Zone = "All",
  AIC = AIC(m),
  BIC = BIC(m),
  R2_marginal = r2[1],
  R2_conditional = r2[2]
)

# =========================================================
# Marginal effects 
# =========================================================
cf <- coef(summary(m))
effects_all <- list()

for (v in vars) {

  if (isTRUE(high_vif[[v]])) {
    effects_all[[length(effects_all) + 1]] <- data.frame(
      Zone = "All",
      Factor = v,

      Early_Effect = NA,
      Early_CI95_L = NA,
      Early_CI95_U = NA,
      Early_p = NA,
      Early_sig = "",

      Late_Effect  = NA,
      Late_CI95_L  = NA,
      Late_CI95_U  = NA,
      Late_p = NA,
      Late_sig = "",

      Delta = NA,
      Delta_CI95_L = NA,
      Delta_CI95_U = NA,
      Delta_p = NA,
      Delta_sig = ""
    )
    next
  }

  vz <- paste0(v, "_z")

  # ---- Early ----
  ame_e <- avg_slopes(
    m, variables = vz,
    newdata = df %>% mutate(Period_bin = 0)
  )

  b_e  <- ame_e$estimate[1]
  p_e  <- ame_e$p.value[1]
  ci_e <- c(ame_e$conf.low[1], ame_e$conf.high[1])

  # ---- Late ----
  ame_l <- avg_slopes(
    m, variables = vz,
    newdata = df %>% mutate(Period_bin = 1)
  )

  b_l  <- ame_l$estimate[1]
  p_l  <- ame_l$p.value[1]
  ci_l <- c(ame_l$conf.low[1], ame_l$conf.high[1])

  # ---- Delta (interaction) ----
  inter1 <- paste0(vz, ":Period_bin")
  inter2 <- paste0("Period_bin:", vz)
  inter  <- if (inter1 %in% rownames(cf)) inter1 else inter2

  delta_est <- cf[inter, "Estimate"]
  delta_se  <- cf[inter, "Std. Error"]
  p_d       <- cf[inter, "Pr(>|t|)"]
  df_d      <- cf[inter, "df"]
  tcrit     <- qt(0.975, df = df_d)

  effects_all[[length(effects_all) + 1]] <- data.frame(
    Zone = "All",
    Factor = v,

    Early_Effect = b_e,
    Early_CI95_L = ci_e[1],
    Early_CI95_U = ci_e[2],
    Early_p = p_e,
    Early_sig = sig_label(p_e),

    Late_Effect  = b_l,
    Late_CI95_L  = ci_l[1],
    Late_CI95_U  = ci_l[2],
    Late_p = p_l,
    Late_sig = sig_label(p_l),

    Delta = b_l - b_e,
    Delta_CI95_L = delta_est - tcrit * delta_se,
    Delta_CI95_U = delta_est + tcrit * delta_se,
    Delta_p = p_d,
    Delta_sig = sig_label(p_d)
  )
}

# =========================================================
# Output
# =========================================================
write_csv(
  bind_rows(effects_all),
  "C:/Users/Lenovo/Phase-specific modelling of 2003/Period_MarginalEffects_All.csv"
)

write_csv(
  model_stats,
  "C:/Users/Lenovo/Phase-specific modelling of 2003/Model_fit_stats_All.csv"
)


Rows: 6834 Columns: 19
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (4): SiteID, Protocol, zone, period
dbl (15): mean_salinity, mean_organic, seg_id_nat, HYBAS_ID, elevation, Long...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Running Richness model: ALL


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif

boundary (singular) fit: see help('isSingular')


Correlation matrix not shown by default, as p = 16 > 12.
Use print(out$value, correlation=TRUE)  or
    vcov(out$value)        if you need it


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `mar

🎉 完成：Richness | ALL only | mixed model + AME + CI + Delta


In [ ]:
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(car)
library(MuMIn)
library(marginaleffects)
library(tibble)

# =========================================================
# Read data
# =========================================================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness_segmented_2005.csv") %>%
  mutate(
    SiteID   = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone),
    period   = as.character(period),
    Period_bin = ifelse(period == "2006_2018", 1, 0)
  )

num_cols <- c(
  "sen_richness","sen_temp","HFP_period",
  "mean_temp","mean_Q","mean_salinity",
  "mean_organic","elevation"
)
df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      sen_richness, sen_temp, HFP_period,
      mean_temp, mean_Q, mean_salinity,
      mean_organic, elevation,
      Protocol, HYBAS_ID, SiteID
    )
  )

# =========================================================
# Standardize predictors
# =========================================================
vars <- c(
  "mean_temp","sen_temp","mean_Q",
  "mean_organic","mean_salinity",
  "HFP_period","elevation"
)

for (v in vars) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[,1]
}

# =========================================================
# Model formula
# =========================================================
fixed_form <- sen_richness ~
  Period_bin +
  mean_temp_z + sen_temp_z + mean_Q_z +
  mean_organic_z + mean_salinity_z +
  HFP_period_z + elevation_z +
  mean_temp_z:Period_bin +
  sen_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_organic_z:Period_bin +
  mean_salinity_z:Period_bin +
  HFP_period_z:Period_bin +
  elevation_z:Period_bin

form <- update(
  fixed_form,
  . ~ . + (1 | HYBAS_ID) + (1 | Protocol) + (1 | SiteID)
)

# =========================================================
# VIF
# =========================================================
core_vars <- c("mean_temp", "sen_temp")
vif_threshold <- 10

get_vif_vals <- function(dat) {
  lm_fit <- lm(fixed_form, data = dat)
  vif_raw <- car::vif(lm_fit)
  if (is.matrix(vif_raw)) {
    vif_raw[, "GVIF^(1/(2*Df))"]
  } else vif_raw
}

is_high_vif <- function(v, vif_vals) {
  if (v %in% core_vars) return(FALSE)
  terms <- c(
    paste0(v, "_z"),
    paste0(v, "_z:Period_bin"),
    paste0("Period_bin:", v, "_z")
  )
  terms <- terms[terms %in% names(vif_vals)]
  any(vif_vals[terms] >= vif_threshold)
}

sig_label <- function(p) {
  if (is.na(p)) "" else if (p < 0.001) "***"
  else if (p < 0.01) "**"
  else if (p < 0.05) "*"
  else ""
}

# =========================================================
# Run model
# =========================================================
cat("Running Richness model: ALL\n")

# ---- VIF ----
vif_vals <- get_vif_vals(df)
vif_table <- data.frame(
  Zone = "All",
  Term = names(vif_vals),
  VIF  = as.numeric(vif_vals)
)

high_vif <- sapply(vars, is_high_vif, vif_vals = vif_vals)
names(high_vif) <- vars

# ---- Mixed model ----
m <- lmer(
  form, data = df, REML = TRUE,
  control = lmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 2e5)
  )
)

# ---- Save summaries ----
writeLines(
  capture.output(summary(m)),
  "C:/Users/Lenovo/Phase-specific modelling of 2005/summary_Richness_All.txt"
)

coef_tab <- coef(summary(m)) %>%
  as.data.frame() %>%
  rownames_to_column("Term")

write_csv(
  coef_tab,
  "C:/Users/Lenovo/Phase-specific modelling of 2005/coef_Richness_All.csv"
)

vc_tab <- as.data.frame(VarCorr(m))
write_csv(
  vc_tab,
  "C:/Users/Lenovo/Phase-specific modelling of 2005/varcomp_Richness_All.csv"
)

r2 <- suppressWarnings(MuMIn::r.squaredGLMM(m))
model_stats <- data.frame(
  Zone = "All",
  AIC = AIC(m),
  BIC = BIC(m),
  R2_marginal = r2[1],
  R2_conditional = r2[2]
)

# =========================================================
# Marginal effects 
# =========================================================
cf <- coef(summary(m))
effects_all <- list()

for (v in vars) {

  if (isTRUE(high_vif[[v]])) {
    effects_all[[length(effects_all) + 1]] <- data.frame(
      Zone = "All",
      Factor = v,

      Early_Effect = NA,
      Early_CI95_L = NA,
      Early_CI95_U = NA,
      Early_p = NA,
      Early_sig = "",

      Late_Effect  = NA,
      Late_CI95_L  = NA,
      Late_CI95_U  = NA,
      Late_p = NA,
      Late_sig = "",

      Delta = NA,
      Delta_CI95_L = NA,
      Delta_CI95_U = NA,
      Delta_p = NA,
      Delta_sig = ""
    )
    next
  }

  vz <- paste0(v, "_z")

  # ---- Early ----
  ame_e <- avg_slopes(
    m, variables = vz,
    newdata = df %>% mutate(Period_bin = 0)
  )

  b_e  <- ame_e$estimate[1]
  p_e  <- ame_e$p.value[1]
  ci_e <- c(ame_e$conf.low[1], ame_e$conf.high[1])

  # ---- Late ----
  ame_l <- avg_slopes(
    m, variables = vz,
    newdata = df %>% mutate(Period_bin = 1)
  )

  b_l  <- ame_l$estimate[1]
  p_l  <- ame_l$p.value[1]
  ci_l <- c(ame_l$conf.low[1], ame_l$conf.high[1])

  # ---- Delta (interaction) ----
  inter1 <- paste0(vz, ":Period_bin")
  inter2 <- paste0("Period_bin:", vz)
  inter  <- if (inter1 %in% rownames(cf)) inter1 else inter2

  delta_est <- cf[inter, "Estimate"]
  delta_se  <- cf[inter, "Std. Error"]
  p_d       <- cf[inter, "Pr(>|t|)"]
  df_d      <- cf[inter, "df"]
  tcrit     <- qt(0.975, df = df_d)

  effects_all[[length(effects_all) + 1]] <- data.frame(
    Zone = "All",
    Factor = v,

    Early_Effect = b_e,
    Early_CI95_L = ci_e[1],
    Early_CI95_U = ci_e[2],
    Early_p = p_e,
    Early_sig = sig_label(p_e),

    Late_Effect  = b_l,
    Late_CI95_L  = ci_l[1],
    Late_CI95_U  = ci_l[2],
    Late_p = p_l,
    Late_sig = sig_label(p_l),

    Delta = b_l - b_e,
    Delta_CI95_L = delta_est - tcrit * delta_se,
    Delta_CI95_U = delta_est + tcrit * delta_se,
    Delta_p = p_d,
    Delta_sig = sig_label(p_d)
  )
}

# =========================================================
# Output
# =========================================================
write_csv(
  bind_rows(effects_all),
  "C:/Users/Lenovo/Phase-specific modelling of 2005/Period_MarginalEffects_All.csv"
)

write_csv(
  model_stats,
  "C:/Users/Lenovo/Phase-specific modelling of 2005/Model_fit_stats_All.csv"
)


Rows: 6834 Columns: 19
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (4): SiteID, Protocol, zone, period
dbl (15): mean_salinity, mean_organic, seg_id_nat, HYBAS_ID, elevation, Long...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Running Richness model: ALL


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif

boundary (singular) fit: see help('isSingular')


Correlation matrix not shown by default, as p = 16 > 12.
Use print(out$value, correlation=TRUE)  or
    vcov(out$value)        if you need it


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `mar